# 05 — Demonstração de telemetria HTTP + SQLite

No teste um servidor HTTP local recebe dados históricos reais, atualiza um detector e grava leituras e alertas em SQLite. O Colab permite testar o HTTP internamente, mas não hospeda automaticamente o painel web para acesso público.

In [1]:
from pathlib import Path
import os, sys, zipfile, subprocess
BASE = Path.cwd()
if not (BASE / 'iot_monitor').exists():
    git_repo = "https://github.com/jorgek14/railpulse-metropt3.git"
    if git_repo.strip():
        subprocess.run(['git', 'clone', git_repo, '/content/railpulse-metropt3'], check=True)
        BASE = Path('/content/railpulse-metropt3')
    else:
        try:
            from google.colab import files
        except ImportError as e:
            raise RuntimeError('Abra o notebook na raiz do repositório ou no Colab e envie o ZIP do projeto.') from e
        print('Selecione o ZIP do repositório RailPulse. NÃO selecione o CSV original agora.')
        enviados = files.upload()
        nome = next((n for n in enviados if n.endswith('.zip')), None)
        assert nome, 'É necessário carregar o ZIP do projeto.'
        BASE = Path('/content/railpulse-metropt3')
        BASE.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(nome) as z:
            z.extractall(BASE)
os.chdir(BASE)
sys.path.insert(0, str(BASE))
print('Raiz do projeto:', BASE)

Raiz do projeto: /content/railpulse-metropt3


In [2]:
import json,threading,tempfile,urllib.request
from pathlib import Path
from http.server import ThreadingHTTPServer
from iot_monitor.server import Context,Handler
from iot_monitor.replay import samples,post_json
PORT=18765
servidor=ThreadingHTTPServer(('127.0.0.1',PORT),Handler)
_tmp=tempfile.TemporaryDirectory()
servidor.ctx=Context(json.loads((BASE/'parametros_etapa4.json').read_text()),Path(_tmp.name)/'railpulse.sqlite')
thread=threading.Thread(target=servidor.serve_forever,daemon=True);thread.start()
print('Servidor local rodando em',PORT)

Servidor local rodando em 18765


In [3]:
leituras=0; avisos=[]
for a in samples(BASE/'dados_demo'/'demo_f3.csv'):
    res=post_json(f'http://127.0.0.1:{PORT}/api/reading',a)
    leituras+=1
    avisos+=res.get('alerts',[])
print('Leituras HTTP:',leituras,'Alertas:',len(avisos))
with urllib.request.urlopen(f'http://127.0.0.1:{PORT}/api/status') as r:
    status=json.load(r)
print('SQLite:',status['counts'])
assert leituras==303 and len(avisos)==3
print('✅ HTTP + SQLite + detecção incremental funcionando.')

Leituras HTTP: 303 Alertas: 3
SQLite: {'readings': 303, 'alerts': 3, 'primary_alerts': 1}
✅ HTTP + SQLite + detecção incremental funcionando.


In [4]:
import pandas as pd
from IPython.display import display
display(pd.DataFrame(avisos)[['timestamp','method','pressure_bar','elapsed_s']])
servidor.shutdown();servidor.server_close();_tmp.cleanup()

,timestamp,method,pressure_bar,elapsed_s
0,2020-06-05 09:51:38,Pressao_nao_recuperada_q99,9.082,188.0
1,2020-06-05 09:51:38,Somente_carga_mesmo_limiar,9.082,188.0
2,2020-06-05 09:54:17,Carga_longa_q99,8.874,347.0
